In [1]:
from sklearn.datasets import make_regression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
from sklearn.datasets import load_diabetes

In [2]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, n_targets=1,noise=20,random_state=13) # 1D data for Simple linear regression
A, b = load_diabetes(return_X_y=True) # 10D data for Multiple linear regression

In [3]:
# GRADIENT DESCENT CLASS FOR SIMPLE LINEAR REGRESSION IMPLEMENTATION
class Batch_Gradient_Descent_slr:
  def __init__(self,learning_rate,epochs):
    self.m = 100 #Starting from a random value of m
    self.b = -120 #Starting from a random value of b
    self.lr = learning_rate
    self.epochs = epochs
    self.X = None
    self.y = None

  def fit(self,X,y):
    self.X = X
    self.y = y
    for i in range(self.epochs):
      #Calculating gradient of 'b' and updating the 'b' value and #Calculating gradient of 'b' and updating the 'b' value
      b_grad = -2 * np.sum(y - self.m*X.ravel() - self.b)
      m_grad = -2 * np.sum(X.ravel() * (y - self.b - self.m*X.ravel()))
      self.b = self.b - (self.lr * b_grad)
      self.m = self.m - (self.lr * m_grad)

  def get_params(self):
    return f"value of b: {self.b} and value of m: {self.m}"

  def compare_params(self):
    LR = LinearRegression()
    LR.fit(self.X,self.y)
    lr_intercept = LR.intercept_
    lr_coef = LR.coef_
    df = pd.DataFrame({
        'intercept': [lr_intercept, self.b],
        'coef': [lr_coef, self.m]
    },index=['sklearn','user-defined-GD_Class'])
    return df

  def predict(self,X):
    return self.m * X + self.b

  def eval_matrix(self, y_true, y_pred):
      LR = LinearRegression()
      LR.fit(self.X,self.y)
      ypred = LR.predict(self.X)
      df = pd.DataFrame({
          'R2_Score': [(r2_score(y_true, y_pred)),(r2_score(y_true, ypred))],
          'Mean Squared Error': [(mean_squared_error(y_true, y_pred)),(mean_squared_error(y_true, ypred))],
          'Mean Absolute Error': [(mean_absolute_error(y_true, y_pred)),(mean_absolute_error(y_true, ypred))]
      },index=['user-defined-gd-class', 'sklearn lr class'])
      return df

In [4]:
bgd = Batch_Gradient_Descent_slr(0.001,100)
bgd.fit(X,y)

In [5]:
bgd.eval_matrix(y,bgd.predict(X))

,R2_Score,Mean Squared Error,Mean Absolute Error
user-defined-gd-class,0.703518,283.422756,13.970042
sklearn lr class,0.703518,283.422756,13.970042


In [6]:
bgd.compare_params()

,intercept,coef
sklearn,-2.294745,[27.82809103252014]
user-defined-GD_Class,-2.294745,27.828092


In [7]:
# GRADIENT DESCENT CLASS FOR MULTIPLE LINEAR REGRESSION IMPLEMENTATION
class Batch_Gradient_Descent_mlr:

  def __init__(self,learning_rate,epochs):
    self.m = None
    self.b = None
    self.lr = learning_rate
    self.epochs = epochs

    # Storing training data internally in the class for further analysis
    self.X_train = None
    self.y_train = None

  def Lr_Outputs(self, X_train, y_train, X_test=None, y_test=None):

    LR = LinearRegression()
    LR.fit(X_train, y_train)

    if X_test is None:
        return LR.intercept_, LR.coef_

    ypred = LR.predict(X_test)
    r2 = r2_score(y_test, ypred)
    mse = mean_squared_error(y_test, ypred)
    mae = mean_absolute_error(y_test, ypred)

    return r2, mse, mae

  def fit(self,X_train,y_train):
    self.X_train = X_train
    self.y_train = y_train

    # Initialize your coefficients
    self.b = 0 #For the 1st time we initialize intercept equals to 0
    self.m = np.ones(X_train.shape[1]) #For the 1st time we initialize all the weights with value 1 and in our case we have 10 columns so there will be 10 weights and therefore we create a array of length 10 each value being 1 in it.

    for i in range(self.epochs):
      # Update all the coefficients and intercept in this loop
      y_hat = np.dot(X_train,self.m) + self.b # we will get y_hat by multiplying two matrices X_train(353,10) and m(10,1) which will result in new matrix of shape (353,1) and then we add intercept value to each of the elements present in the new matrix
      b_derivative = -2 * np.mean(y_train - y_hat) # y_train is original values and y_hat is predicted value
      self.b = self.b - (self.lr * b_derivative)

      m_derivatives =  -2 * np.dot((y_train - y_hat), X_train)/X_train.shape[0]
      self.m = self.m  - (self.lr * m_derivatives)

  def predict(self,X_test):
    return np.dot(X_test,self.m) + self.b

  def get_params(self):
    return self.b, self.m

  def compare_params(self):
    lr_intercept, lr_coef = self.Lr_Outputs(self.X_train,self.y_train)
    print(f'[Sklearn LR class results] -> intercept: {lr_intercept} | coef: {lr_coef}')
    print("\n")
    print(f'[User-defined GD class results] -> intercept: {self.b} | coef: {self.m}')

  def eval_matrix(self, X_test, y_test):

    # Get sklearn metrics using existing method
    r2_sklearn, mse_sklearn, mae_sklearn = self.Lr_Outputs(
        self.X_train, self.y_train, X_test, y_test
    )

    # Get user-defined GD predictions
    y_pred_gd = self.predict(X_test)

    r2_gd = r2_score(y_test, y_pred_gd)
    mse_gd = mean_squared_error(y_test, y_pred_gd)
    mae_gd = mean_absolute_error(y_test, y_pred_gd)

    df = pd.DataFrame({
        'R2_Score': [r2_sklearn, r2_gd],
        'Mean Squared Error': [mse_sklearn, mse_gd],
        'Mean Absolute Error': [mae_sklearn, mae_gd]
    }, index=['sklearn_lr_class', 'user_defined_GD_class'])

    return df


In [32]:
class Stochastic_Gradient_Descent:
  def __init__(self, epochs, lr):
    self.b = None;
    self.m = None;
    self.epochs = epochs
    self.lr = lr
    self.X_train = None
    self.y_train = None

  def __Lr_Outputs(self, X_train, y_train, X_test=None, y_test=None):
    LR = LinearRegression()
    LR.fit(X_train, y_train)

    if X_test is None:
        return LR.intercept_, LR.coef_
    ypred = LR.predict(X_test)
    r2 = r2_score(y_test, ypred)
    mse = mean_squared_error(y_test, ypred)
    mae = mean_absolute_error(y_test, ypred)

    return r2, mse, mae

  def fit(self, X_train, y_train):
    self.b = 0
    self.m = np.ones(X_train.shape[1])
    self.X_train = X_train
    self.y_train = y_train
    for i in range(self.epochs):
      for  j in range(X_train.shape[0]):
        random_index = np.random.randint(0, X_train.shape[0])
        y_hat = np.dot(X_train[random_index],self.m) + self.b
        b_derivative = -2 * (y_train[random_index] - y_hat)
        self.b = self.b - (self.lr * b_derivative)

        m_derivatives =  -2 * np.dot((y_train[random_index] - y_hat), X_train[random_index])
        self.m = self.m - (self.lr * m_derivatives)

  def predict(self,X_test):
      return np.dot(X_test,self.m) + self.b

  def eval_matrix(self, X_test, y_test, y_pred):
      # Get sklearn metrics using existing method
      r2_sklearn, mse_sklearn, mae_sklearn = self.__Lr_Outputs(
          self.X_train, self.y_train, X_test, y_test
      )

      # Get user-defined GD predictions
      r2_gd = r2_score(y_test, y_pred)
      mse_gd = mean_squared_error(y_test, y_pred)
      mae_gd = mean_absolute_error(y_test, y_pred)

      df = pd.DataFrame({
          'R2_Score': [r2_sklearn, r2_gd],
          'Mean Squared Error': [mse_sklearn, mse_gd],
          'Mean Absolute Error': [mae_sklearn, mae_gd]
      }, index=['sklearn_lr_class', 'user_defined_GD_class'])

      return df

  def compare_params(self):
      lr_intercept, lr_coef = self.__Lr_Outputs(self.X_train,self.y_train)
      print(f'[Sklearn LR class results] -> intercept: {lr_intercept} | coef: {lr_coef}')
      print("\n")
      print(f'[User-defined GD class results] -> intercept: {self.b} | coef: {self.m}')

In [51]:
X_train, X_test, y_train, y_test = train_test_split(A, b, test_size=0.2)

In [62]:
sgd = Stochastic_Gradient_Descent(40,0.1)
sgd.fit(X_train,y_train)
y_pred = sgd.predict(X_test)
sgd.eval_matrix(X_test,y_test, y_pred)

,R2_Score,Mean Squared Error,Mean Absolute Error
sklearn_lr_class,0.475405,2841.666879,43.539556
user_defined_GD_class,0.351522,3512.722433,47.563366


In [63]:
sgd.compare_params()

[Sklearn LR class results] -> intercept: 150.83132504605936 | coef: [  -19.19314295  -182.96927872   515.71117529   346.04497992
 -1069.86801827   684.76367253   243.67643502   180.6244999
   901.19637559    63.88494998]


[User-defined GD class results] -> intercept: 128.1598531927034 | coef: [ -22.11992808 -206.96839646  564.14799101  332.62973827  -77.65168875
 -127.22025281 -186.81886276  107.94145221  526.17148957   44.47965863]
